# 🔬 Retinal OCT — ResNet-50
### Reproducible 80/20 Train–Val Split · Locked Test Set

| Item | Value |
|------|-------|
| Dataset | Kaggle Retinal OCT (~84,495 images) |
| Classes | CNV · DME · DRUSEN · NORMAL |
| Split | 80 % train / 20 % val (stratified, from `train/` folder) |
| Test set | Official `test/` folder — **never touched during training** |
| Random seed | `SEED = 42` — frozen everywhere |
| Strategy | Frozen backbone (ep 1–4) → full fine-tune (ep 5+) |

---

## 0. Environment & Reproducibility Setup

In [ ]:
!pip install torch torchvision scikit-learn matplotlib seaborn --quiet

import os, random, json, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torch.cuda.amp import GradScaler, autocast
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED);  np.random.seed(SEED)
torch.manual_seed(SEED);  torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f'PyTorch : {torch.__version__}  |  Seed : {SEED}')

## 1. Configuration

In [ ]:
DATA_DIR       = './data eye/OCT2017'
BATCH_SIZE     = 64
NUM_EPOCHS     = 20
LR_HEAD        = 1e-3
LR_BACKBONE    = 1e-4
UNFREEZE_EPOCH = 5
VAL_SPLIT      = 0.20
NUM_CLASSES    = 4
IMG_SIZE       = 224
NUM_WORKERS    = 0   # 0 = main process only (required on Windows)
CLASS_NAMES    = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
SAVE_PATH      = 'best_resnet50.pth'
LOG_PATH       = 'resnet50_history.json'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

for split in ['train', 'test']:
    p = os.path.join(DATA_DIR, split)
    print(f'  {"✓" if os.path.exists(p) else "✗ MISSING"}  {p}')

## 2. Stratified 80/20 Split

> The Kaggle OCT `train/` folder contains ~83,484 images.  
> We apply `StratifiedShuffleSplit` to create a reproducible 80/20 partition.  
> The official `test/` set (968 images) is **locked** — opened only in Section 11.

In [ ]:
base_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
aug_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

full_ds  = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=base_tf)
aug_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=aug_tf)
targets  = np.array(full_ds.targets)

sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SPLIT, random_state=SEED)
train_idx, val_idx = next(sss.split(np.arange(len(full_ds)), targets))

assert len(set(train_idx) & set(val_idx)) == 0, 'LEAKAGE DETECTED'

print(f'Total   : {len(full_ds):,}  |  Train: {len(train_idx):,}  |  Val: {len(val_idx):,}')
print(f'Leakage check : ✓')
print('\nPer-class distribution:')
for i, n in enumerate(CLASS_NAMES):
    tr = (targets[train_idx]==i).sum()
    va = (targets[val_idx]  ==i).sum()
    print(f'  {n:<8} train={tr:>5,}  val={va:>4,}  ({tr/(tr+va)*100:.1f}/{va/(tr+va)*100:.1f})')

In [ ]:
# Windows-safe worker init — lambda functions cannot be pickled by multiprocessing
def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

# Use NUM_WORKERS=0 on Windows if you still get PicklingError
# NUM_WORKERS = 0
print(f'Worker init function defined. NUM_WORKERS={NUM_WORKERS}')

In [ ]:
g = torch.Generator(); g.manual_seed(SEED)

train_loader = DataLoader(Subset(aug_ds,  train_idx), batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True,
                          generator=g)
val_loader   = DataLoader(Subset(full_ds, val_idx),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(
    datasets.ImageFolder(os.path.join(DATA_DIR,'test'), transform=base_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train  : {len(train_loader.dataset):,}  Val : {len(val_loader.dataset):,}  Test : {len(test_loader.dataset):,} [LOCKED]')

## 3. Class Balance Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
for ax, name, idx in [(axes[0],'Train (80%)',train_idx),(axes[1],'Val (20%)',val_idx)]:
    counts = [(targets[idx]==i).sum() for i in range(NUM_CLASSES)]
    bars = ax.bar(CLASS_NAMES, counts, color=['#4C72B0','#DD8452','#55A868','#C44E52'])
    ax.bar_label(bars, fmt='%d'); ax.set_title(name); ax.set_ylabel('Count')
plt.suptitle('ResNet-50 — Stratified Class Balance', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('resnet50_class_balance.png', dpi=150); plt.show()

## 4. Build ResNet-50 Model

In [ ]:
torch.manual_seed(SEED)
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

for param in model.parameters():
    param.requires_grad = False        # freeze backbone

in_f = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_f, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True), nn.Dropout(0.4),
    nn.Linear(512, 256),  nn.ReLU(inplace=True), nn.Dropout(0.3),
    nn.Linear(256, NUM_CLASSES),
)
model = model.to(DEVICE)
print(f'Head-only trainable params : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 5. Loss, Optimiser & Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR_HEAD, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler    = GradScaler()
print('Ready.')

## 6. Unfreeze Helper & run_epoch

In [ ]:
def unfreeze_backbone(model, optimizer):
    for p in model.parameters(): p.requires_grad = True
    bb = [p for n,p in model.named_parameters() if 'fc' not in n]
    optimizer.add_param_group({'params': bb, 'lr': LR_BACKBONE})
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  ✓ Backbone unfrozen | Trainable params: {total:,}')

def run_epoch(model, loader, optimizer, criterion, scaler, training=True):
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training: optimizer.zero_grad()
            with autocast():
                logits = model(imgs)
                loss   = criterion(logits, labels)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            probs = torch.softmax(logits.detach(), dim=1)
            preds = probs.argmax(1)
            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return (total_loss/total, correct/total,
            balanced_accuracy_score(all_labels, all_preds),
            all_preds, all_labels, np.array(all_probs))

print('Helpers defined.')

## 7. Training Loop

In [ ]:
history = {k:[] for k in ['train_loss','val_loss','train_acc','val_acc',
                            'train_bal_acc','val_bal_acc','lr']}
best_val_acc = 0.0
unfrozen = False

for epoch in range(1, NUM_EPOCHS + 1):
    if epoch == UNFREEZE_EPOCH and not unfrozen:
        print(f'--- Epoch {epoch}: Unfreezing backbone ---')
        unfreeze_backbone(model, optimizer); unfrozen = True

    tr = run_epoch(model, train_loader, optimizer, criterion, scaler, True)
    va = run_epoch(model, val_loader,   optimizer, criterion, scaler, False)
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    for k, v in zip(['train_loss','train_acc','train_bal_acc'], [tr[0],tr[1],tr[2]]): history[k].append(v)
    for k, v in zip(['val_loss',  'val_acc',  'val_bal_acc'],   [va[0],va[1],va[2]]): history[k].append(v)
    history['lr'].append(lr)

    print(f'Epoch [{epoch:02d}/{NUM_EPOCHS}]  '
          f'Train Loss:{tr[0]:.4f} Acc:{tr[1]:.4f} Bal:{tr[2]:.4f}  |  '
          f'Val   Loss:{va[0]:.4f} Acc:{va[1]:.4f} Bal:{va[2]:.4f}  LR:{lr:.2e}')

    if va[1] > best_val_acc:
        best_val_acc = va[1]
        torch.save({'epoch':epoch,'model_state':model.state_dict(),
                    'val_acc':va[1],'seed':SEED}, SAVE_PATH)
        print(f'  ✓ Saved (val_acc={va[1]:.4f})')

with open(LOG_PATH,'w') as f: json.dump(history, f, indent=2)
print(f'\nBest val acc: {best_val_acc:.4f}')

## 8. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))
axes[0].plot(history['train_loss'], label='Train', lw=2)
axes[0].plot(history['val_loss'],   label='Val',   lw=2)
axes[0].axvline(UNFREEZE_EPOCH-1, color='red', ls='--', label='Unfreeze')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'],     label='Train Acc',     lw=2)
axes[1].plot(history['val_acc'],       label='Val Acc',       lw=2)
axes[1].plot(history['train_bal_acc'], label='Train Bal-Acc', lw=2, ls='--')
axes[1].plot(history['val_bal_acc'],   label='Val Bal-Acc',   lw=2, ls='--')
axes[1].axvline(UNFREEZE_EPOCH-1, color='red', ls='--', label='Unfreeze')
axes[1].set_title('Accuracy'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

axes[2].plot(history['lr'], lw=2, color='green')
axes[2].set_title('LR Schedule'); axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.suptitle('ResNet-50 — 80/20 Split Training History', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.savefig('resnet50_training_curves.png', dpi=150); plt.show()

## 9. Overfitting Diagnostic

In [ ]:
gap = history['train_acc'][-1] - history['val_acc'][-1]
print(f'Train acc: {history["train_acc"][-1]:.4f}')
print(f'Val   acc: {history["val_acc"][-1]:.4f}')
print(f'Gap      : {gap:.4f}')
print('✓ Good' if gap <= 0.05 else ('⚠ Moderate overfit' if gap <= 0.10 else '⚠ High overfit'))
if history['val_acc'][-1] > 0.999:
    print('⚠ 100% val acc — check for data leakage!')

## 10. Validation Statistical Report

In [ ]:
ckpt = torch.load(SAVE_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded epoch {ckpt["epoch"]} (val_acc={ckpt["val_acc"]:.4f})')

_, va_acc, va_bal, va_preds, va_labels, va_probs = run_epoch(
    model, val_loader, None, criterion, None, False)

try:
    auc = roc_auc_score(np.eye(NUM_CLASSES)[va_labels], va_probs, multi_class='ovr', average='macro')
    print(f'Val AUC-ROC (macro OvR): {auc:.4f}')
except Exception as e: print(f'AUC: {e}')
print(f'Val Accuracy         : {va_acc:.4f}')
print(f'Val Balanced Accuracy: {va_bal:.4f}')
print(classification_report(va_labels, va_preds, target_names=CLASS_NAMES, digits=4))

## 11. 🔒 Final Test Set Evaluation (Run Only Once)

In [ ]:
_, te_acc, te_bal, te_preds, te_labels, te_probs = run_epoch(
    model, test_loader, None, criterion, None, False)
try:
    te_auc = roc_auc_score(np.eye(NUM_CLASSES)[te_labels], te_probs, multi_class='ovr', average='macro')
    print(f'Test AUC-ROC: {te_auc:.4f}')
except Exception as e: print(f'AUC: {e}')
print(f'Test Accuracy          : {te_acc:.4f}')
print(f'Test Balanced Accuracy : {te_bal:.4f}')
print(classification_report(te_labels, te_preds, target_names=CLASS_NAMES, digits=4))

## 12. Confusion Matrices — Val & Test

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,6))
for ax,preds,labels,title,cmap in [
    (axes[0], va_preds, va_labels, 'Val (20%) Confusion Matrix',   'Oranges'),
    (axes[1], te_preds, te_labels, 'Test (Locked) Confusion Matrix','Purples'),
]:
    sns.heatmap(confusion_matrix(labels,preds), annot=True, fmt='d', cmap=cmap,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, linewidths=0.5, ax=ax)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.tight_layout(); plt.savefig('resnet50_confusion_matrices.png', dpi=150); plt.show()

## 13. Results Summary

In [ ]:
print('=' * 55)
print('  RESNET-50 — FINAL RESULTS SUMMARY')
print('=' * 55)
print(f'  Seed              : {SEED}')
print(f'  Split             : 80/20 stratified')
print(f'  Best Val Accuracy : {best_val_acc:.4f}')
print(f'  Val  Balanced Acc : {va_bal:.4f}')
print(f'  Test Accuracy     : {te_acc:.4f}')
print(f'  Test Balanced Acc : {te_bal:.4f}')
print('=' * 55)
print('For publication: run SEED in [42,0,1,7,123] → report mean ± std')